# Sydney Rainfall Prediction Using Deep Learning

This notebook predicts **rainy days in Sydney** from historical weather observations and compares two neural-network approaches:

1. **Dense Neural Network:** predicts whether the current day is rainy from the day's weather variables, excluding measured rainfall.
2. **LSTM:** uses the previous seven days of weather observations to predict whether the following day is rainy.

A rainy day is defined as daily rainfall of at least **1.0 mm**. All figures are saved automatically in the repository's `images/` folder.

## 1. Setup and imports

Run this notebook from the repository root with the `waterlab` kernel selected.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

try:
    import tensorflow as tf
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.utils import plot_model
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is required for the model sections. Activate the waterlab "
        "environment and run: python -m pip install tensorflow"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "weather_sydney.csv"
IMAGES_DIR = PROJECT_ROOT / "images"
MODELS_DIR = PROJECT_ROOT / "models"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

## 2. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Place weather_sydney.csv inside the data folder."
    )

weather = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
weather.index = pd.to_datetime(weather.index)
weather = weather.sort_index()
weather = weather[~weather.index.duplicated(keep="first")]

print(f"Rows: {weather.shape[0]:,}")
print(f"Columns: {weather.shape[1]}")
print(f"Date range: {weather.index.min().date()} to {weather.index.max().date()}")
display(weather.head())

In [ ]:
weather.info()
display(weather.describe(include="all").T)

## 3. Data cleaning and exploratory analysis

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": weather.isna().sum(),
    "missing_percent": weather.isna().mean().mul(100),
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

plt.figure(figsize=(12, 6))
missing_summary.loc[missing_summary["missing_count"] > 0, "missing_percent"].plot(kind="bar")
plt.title("Missing Values by Weather Variable")
plt.xlabel("Weather variable")
plt.ylabel("Missing values (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "missing_values.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Interpolate numeric measurements in chronological order.
numeric_columns = weather.select_dtypes(include=np.number).columns
weather[numeric_columns] = (
    weather[numeric_columns]
    .interpolate(method="time", limit_direction="both")
    .ffill()
    .bfill()
)

# Fill any categorical gaps with the most frequent category.
for column in weather.select_dtypes(exclude=np.number).columns:
    mode = weather[column].mode(dropna=True)
    if not mode.empty:
        weather[column] = weather[column].fillna(mode.iloc[0])

print("Remaining missing values:", int(weather.isna().sum().sum()))

In [ ]:
if "Rainfall" not in weather.columns:
    raise KeyError("The dataset must contain a 'Rainfall' column.")

weather["Rainy_Day"] = (weather["Rainfall"] >= 1.0).astype(int)

rainy_counts = weather["Rainy_Day"].value_counts().sort_index()
print(rainy_counts.rename(index={0: "No rain", 1: "Rain"}))

plt.figure(figsize=(7, 5))
sns.countplot(data=weather, x="Rainy_Day")
plt.title("Distribution of Rainy and Non-Rainy Days")
plt.xlabel("Rainy day (0 = No, 1 = Yes)")
plt.ylabel("Number of days")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "target_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=weather, x="Rainfall", bins=40, kde=True)
plt.title("Distribution of Daily Rainfall in Sydney")
plt.xlabel("Daily rainfall (mm)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "rainfall_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

monthly_rainfall = weather["Rainfall"].resample("ME").sum()
plt.figure(figsize=(14, 6))
monthly_rainfall.plot(linewidth=1)
plt.title("Monthly Rainfall Totals in Sydney")
plt.xlabel("Date")
plt.ylabel("Monthly rainfall (mm)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "monthly_rainfall.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
numeric_weather = weather.select_dtypes(include=np.number)
plt.figure(figsize=(15, 11))
sns.heatmap(numeric_weather.corr(), cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Correlation Matrix of Sydney Weather Variables")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Feature preparation and chronological split

`Rainfall` is excluded from the predictors because it is used to define the target. The split is chronological to avoid leaking future observations into training.

In [ ]:
# Use numeric predictors only, excluding rainfall and the derived target.
feature_columns = [
    column for column in weather.select_dtypes(include=np.number).columns
    if column not in {"Rainfall", "Rainy_Day"}
]

model_data = weather[feature_columns + ["Rainy_Day"]].dropna().copy()

# Preserve the assignment's original cutoff where possible.
requested_cutoff = pd.Timestamp("2017-01-01")
if model_data.index.min() < requested_cutoff < model_data.index.max():
    cutoff = requested_cutoff
else:
    cutoff = model_data.index[int(len(model_data) * 0.80)]

train_data = model_data.loc[model_data.index < cutoff]
test_data = model_data.loc[model_data.index >= cutoff]

X_train = train_data[feature_columns]
y_train = train_data["Rainy_Day"].astype(int)
X_test = test_data[feature_columns]
y_test = test_data["Rainy_Day"].astype(int)

if X_train.empty or X_test.empty:
    raise ValueError("The chronological train/test split produced an empty partition.")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Features:", feature_columns)
print("Training period:", X_train.index.min(), "to", X_train.index.max())
print("Testing period:", X_test.index.min(), "to", X_test.index.max())
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

## 5. Dense Neural Network

In [ ]:
dense_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],), name="weather_features"),
    Dense(128, activation="relu", name="dense_128"),
    Dropout(0.20, name="dropout_1"),
    Dense(64, activation="relu", name="dense_64"),
    Dropout(0.20, name="dropout_2"),
    Dense(1, activation="sigmoid", name="rain_probability"),
], name="dense_rainfall_classifier")

dense_model.compile(
    optimizer="adam",
    loss="binary_focal_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

dense_model.summary()

try:
    plot_model(
        dense_model,
        to_file=str(IMAGES_DIR / "dense_model_architecture.png"),
        show_shapes=True,
        show_layer_names=True,
        dpi=150,
    )
except (ImportError, OSError) as exc:
    print("Model diagram skipped. Install Graphviz/pydot to create it:", exc)

In [ ]:
dense_early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

dense_history = dense_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[dense_early_stopping],
    verbose=1,
)

history = pd.DataFrame(dense_history.history)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(history.index + 1, history["loss"], label="Training loss")
ax.plot(history.index + 1, history["val_loss"], label="Validation loss")
ax.set(title="Dense Model Training History", xlabel="Epoch", ylabel="Loss")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "dense_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(history.index + 1, history["accuracy"], label="Training accuracy")
ax.plot(history.index + 1, history["val_accuracy"], label="Validation accuracy")
ax.set(title="Dense Model Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "dense_training_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
dense_probabilities = dense_model.predict(X_test_scaled, verbose=0).ravel()
dense_predictions = (dense_probabilities >= 0.5).astype(int)

print(classification_report(y_test, dense_predictions, target_names=["No rain", "Rain"], digits=3))

dense_metrics = {
    "Model": "Dense Neural Network",
    "Accuracy": accuracy_score(y_test, dense_predictions),
    "Precision": precision_score(y_test, dense_predictions, zero_division=0),
    "Recall": recall_score(y_test, dense_predictions, zero_division=0),
    "F1": f1_score(y_test, dense_predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, dense_probabilities),
}

cm = confusion_matrix(y_test, dense_predictions)
display_cm = ConfusionMatrixDisplay(cm, display_labels=["No rain", "Rain"])
display_cm.plot(cmap="Blues", values_format="d")
plt.title("Dense Model Confusion Matrix")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "dense_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. LSTM model using the previous seven days

Each LSTM sample contains seven consecutive days of weather predictors. Its target is the rainy-day label of the next day.

In [ ]:
def create_sequences(features, targets, sequence_length=7):
    sequences = []
    labels = []
    for end_index in range(sequence_length, len(features)):
        start_index = end_index - sequence_length
        sequences.append(features[start_index:end_index])
        labels.append(targets[end_index])
    return np.asarray(sequences, dtype="float32"), np.asarray(labels, dtype="int32")

SEQUENCE_LENGTH = 7
X_train_lstm, y_train_lstm = create_sequences(
    X_train_scaled, y_train.to_numpy(), SEQUENCE_LENGTH
)
X_test_lstm, y_test_lstm = create_sequences(
    X_test_scaled, y_test.to_numpy(), SEQUENCE_LENGTH
)

print("LSTM training shape:", X_train_lstm.shape)
print("LSTM testing shape:", X_test_lstm.shape)

In [ ]:
lstm_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, X_train_lstm.shape[2]), name="weather_sequence"),
    LSTM(128, return_sequences=True, name="lstm_128_1"),
    Dropout(0.20, name="dropout_1"),
    LSTM(64, name="lstm_64"),
    Dropout(0.20, name="dropout_2"),
    Dense(1, activation="sigmoid", name="rain_probability"),
], name="lstm_rainfall_classifier")

lstm_model.compile(
    optimizer="adam",
    loss="binary_focal_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

lstm_model.summary()

try:
    plot_model(
        lstm_model,
        to_file=str(IMAGES_DIR / "lstm_model_architecture.png"),
        show_shapes=True,
        show_layer_names=True,
        dpi=150,
    )
except (ImportError, OSError) as exc:
    print("Model diagram skipped. Install Graphviz/pydot to create it:", exc)

In [ ]:
lstm_early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

lstm_history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[lstm_early_stopping],
    verbose=1,
)

lstm_history_df = pd.DataFrame(lstm_history.history)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lstm_history_df.index + 1, lstm_history_df["loss"], label="Training loss")
ax.plot(lstm_history_df.index + 1, lstm_history_df["val_loss"], label="Validation loss")
ax.set(title="LSTM Training History", xlabel="Epoch", ylabel="Loss")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "lstm_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lstm_history_df.index + 1, lstm_history_df["accuracy"], label="Training accuracy")
ax.plot(lstm_history_df.index + 1, lstm_history_df["val_accuracy"], label="Validation accuracy")
ax.set(title="LSTM Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "lstm_training_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lstm_probabilities = lstm_model.predict(X_test_lstm, verbose=0).ravel()
lstm_predictions = (lstm_probabilities >= 0.5).astype(int)

print(classification_report(y_test_lstm, lstm_predictions, target_names=["No rain", "Rain"], digits=3))

lstm_metrics = {
    "Model": "LSTM (7-day sequence)",
    "Accuracy": accuracy_score(y_test_lstm, lstm_predictions),
    "Precision": precision_score(y_test_lstm, lstm_predictions, zero_division=0),
    "Recall": recall_score(y_test_lstm, lstm_predictions, zero_division=0),
    "F1": f1_score(y_test_lstm, lstm_predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test_lstm, lstm_probabilities),
}

cm = confusion_matrix(y_test_lstm, lstm_predictions)
display_cm = ConfusionMatrixDisplay(cm, display_labels=["No rain", "Rain"])
display_cm.plot(cmap="Blues", values_format="d")
plt.title("LSTM Confusion Matrix")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "lstm_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Model comparison and saved outputs

In [ ]:
model_comparison = pd.DataFrame([dense_metrics, lstm_metrics]).set_index("Model")
display(model_comparison.round(3))
model_comparison.to_csv(PROJECT_ROOT / "model_comparison.csv")

comparison_long = (
    model_comparison[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .reset_index()
    .melt(id_vars="Model", var_name="Metric", value_name="Score")
)

plt.figure(figsize=(11, 6))
sns.barplot(data=comparison_long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1)
plt.title("Dense and LSTM Model Performance")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Save models in the modern Keras format.
dense_model.save(MODELS_DIR / "dense_rainfall_model.keras")
lstm_model.save(MODELS_DIR / "lstm_rainfall_model.keras")

print("Generated images:")
for image_path in sorted(IMAGES_DIR.glob("*.png")):
    print(f"- {image_path.name} ({image_path.stat().st_size / 1024:.1f} KB)")

## 8. Conclusion

This workflow provides a reproducible comparison between a feed-forward neural network and a sequence-based LSTM model for rainy-day classification. The final comparison table should be used to determine which model performs better; conclusions should be based on test-set precision, recall, F1 score, ROC-AUC, and not accuracy alone.

Possible next improvements include class weighting, threshold tuning, hyperparameter optimization, and comparison with tree-based baselines such as Random Forest or gradient boosting.